## 🧭 Notebook Map (what each section does)

This notebook is a guided walkthrough of building a **matchmaking search engine**
with Pinecone (a managed vector database). Read it top to bottom; each section
solves one real production problem.

| Section | What it covers |
|---|---|
| **A. Setup & Data** | Install Pinecone, connect with an API key, generate 400 fake dating profiles |
| **B. Build the Dense Index** | Create a vector index and upload ("upsert") the profiles into it |
| **C. Basic Semantic Search** | Search by *meaning* (dense/vector search) and see where it fails on exact facts |
| **D. Fix #1 — Metadata Filters** | Use filters for exact facts (college, batch, language, area) instead of embeddings |
| **E. Keyword Search (Sparse Index)** | Add a BM25-style keyword index to catch exact-word matches |
| **F. Hybrid Search** | Combine dense + sparse results using Reciprocal Rank Fusion (RRF) |
| **G. Scaling — Compression** | Understand vector quantization (why/how Pinecone shrinks memory use) |
| **H. Evaluation Metrics** | Measure search quality with Precision, Recall, MRR, MAP, NDCG |
| **I. Reranking** | Add a second-pass reranker to fix poor ranking (e.g., rank 12 → top 3) |
| **J. Production Ops** | Measure latency percentiles (p50/p95/p99) and add a simple cache |
| **K. Wrap-up** | Recap + cleanup (delete indexes to avoid charges) |


# Session 4 · Vector Databases 2 — **DilSeVector, Production Edition** 💘
### The code a real 2026 engineering team actually ships. Text in → matches out. You will never touch a vector today.

**The rule of this notebook: every fix is small.** Watch the line counts:
- 🎫#2 exact facts → **1 line** (a metadata filter) + a sparse index for keyword relevance
- 🎫#3 rank-12 match → **4 lines** (a `rerank=` parameter on the SAME search call)
- 🎫#1 300 GB RAM → **0 lines** (managed compression — we do the math, Pinecone does the work)
- 🎫#4 launch night → **~10 lines** (a timer + a cache dict; the real dashboard is the console)

**Before class:** run Cells 0–3 (index creation takes ~1 min, upsert+embedding ~1 min). Everything else is live.
**During class:** every 🎲 CHAT BET gets pasted into Zoom chat BEFORE running the cell. Never run in silence.


# 🅰️ SECTION A — SETUP & DATA

## Cell 0 — Setup (~1 minute, run before class)
One dependency. One key. That's the whole stack.

In [8]:
# Install the Pinecone Python client (quiet mode: -q hides the pip logs)
!pip -q install pinecone

from pinecone import Pinecone   # the client class we use to talk to the Pinecone service
import time                     # used later for polling / sleeping while things are created
from google.colab import userdata

# Create a client connected to your Pinecone account.
# Replace "YOUR_API_KEY" with the real key from app.pinecone.io.
pinecone_key = userdata.get('PINECONE_API_KEY')
if not pinecone_key:
    raise RuntimeError(
        "PINECONE_API_KEY not found. "
        "Add it via Colab → 🔑 Secrets → enable Notebook access."
    )
print(repr(pinecone_key))
pc = Pinecone(api_key=pinecone_key)   # from app.pinecone.io
print("Connected ✅")


'pcsk_3SVEBj_Nhc3LfY6vt8NL47MhdBwTjyFvLLQMZyZxco1wGwn8Nc6XcKGxmUGEuEyPDY751T'
Connected ✅


## Cell 1 — The user base: 400 profiles
Plain Python dicts — this is just *data*, the kind any app has. Three profiles are **planted** so every demo has a guaranteed payoff:
- **P0001** — the exact profile Crisis #2's user wants (IIT Bombay · 2019 · Marathi · Dadar)
- **P0002** — the paraphrase twin: "sorted but chill" (zero words shared with "fun-loving but settled")
- **P0003** — the golden match for the rerank story (non-smoker · golden retriever · Dadar)

In [3]:
import random
random.seed(42)   # fixed seed => the "random" profiles are identical every time this cell runs

# Pools of values used to randomly build each fake profile
LOCALITIES = ["Dadar","Thane","Byculla","Dombivli","South Bombay","Bandra","Andheri","Powai"]
COLLEGES   = ["IIT Bombay","VJTI","NMIMS","St Xaviers","Mumbai University","SPIT","ICT Mumbai"]
JOBS       = ["software engineer at TCS","chartered accountant","doctor","startup founder",
              "graphic designer","data scientist","teacher","investment banker","architect"]
VIBES = ["adventurous and loves trekking in the Sahyadris",
         "fun-loving but settled, enjoys quiet weekends",
         "foodie who hunts for the best vada pav in the city",
         "bookworm, poetry evenings at Prithvi cafe",
         "gym at 6am, marathon runner, discipline is life",
         "stand-up comedy fan, drags friends to open mics",
         "loves dogs, volunteers at animal shelters on weekends",
         "sorted but chill, big on family, small on drama",
         "travel junkie, 14 countries and counting",
         "classical music and long walks at Marine Drive"]
LANGS = ["Marathi","Gujarati","Hindi","Konkani","Tamil"]

# Build 400 random profiles as plain Python dictionaries.
# Each dict is one "record" that will later be uploaded ("upserted") to Pinecone.
users = []
for i in range(400):
    u = {"_id": f"P{i:04d}", "locality": random.choice(LOCALITIES), "college": random.choice(COLLEGES),
         "batch": random.choice(range(2014, 2023)), "job": random.choice(JOBS),
         "vibe": random.choice(VIBES), "lang": random.choice(LANGS),
         "smoke": random.choice(["non-smoker", "occasional smoker"])}
    users.append(u)

# --- Planted demo profiles ---------------------------------------------------
# These 3 profiles are hand-edited afterwards so later demos have a guaranteed,
# predictable result to point at.

# planted demo profiles
# P0001 (index 1): the "perfect" profile that exactly matches the biodata-style
# query used later — IIT Bombay, batch 2019, Marathi, lives in Dadar, non-smoker.
users[1].update(college="IIT Bombay", batch=2019, lang="Marathi", locality="Dadar", smoke="non-smoker",
                vibe="loves dogs, volunteers at animal shelters on weekends", job="data scientist")

# P0002 (index 2): a "paraphrase twin" — means almost the same thing as another
# query but shares ZERO exact words with it ("sorted but chill" vs "fun-loving
# but settled"). Used to show dense/keyword search have different blind spots.
users[2].update(vibe="sorted but chill, big on family, small on drama", locality="Bandra")

# P0003 (index 3): the "golden match" used later to demonstrate reranking.
users[3].update(smoke="non-smoker", locality="Dadar", job="architect",
                vibe="loves dogs, has a golden retriever named Simba")

# Build a human-readable "bio" string for each user. This is the text field
# that Pinecone will actually turn into a vector (embed), and it's also what
# the app would display on a profile card.
for u in users:   # the bio is what the app shows on a profile card
    u["bio"] = (f"{u['job'].capitalize()}, {u['college']} {u['batch']} batch. {u['lang']} speaking, "
                f"lives in {u['locality']}. {u['vibe'].capitalize()}. {u['smoke'].capitalize()}.")

# Sanity check: print the 3 planted profiles + total count
for i in (1, 2, 3): print(users[i]["_id"], "→", users[i]["bio"])
print("Total users:", len(users))


P0001 → Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal shelters on weekends. Non-smoker.
P0002 → Architect, Mumbai University 2014 batch. Tamil speaking, lives in Bandra. Sorted but chill, big on family, small on drama. Occasional smoker.
P0003 → Architect, St Xaviers 2018 batch. Konkani speaking, lives in Dadar. Loves dogs, has a golden retriever named simba. Non-smoker.
Total users: 400


# 🅱️ SECTION B — BUILD THE DENSE (SEMANTIC) SEARCH INDEX

## Cell 2 — Create the index (this is the whole "embedding pipeline")
**Say:** *"Notice what's NOT here: no model download, no `.encode()`, no numpy. We tell Pinecone which hosted embedding model to use and WHICH FIELD to embed. Done. This is how production teams ship."*

In [9]:
DENSE = "dilse-dense"   # name of our semantic ("dense vector") index

# Only create the index if it doesn't already exist (avoids an error on re-run)
existing_indexes = [i.name for i in pc.list_indexes()]
if DENSE not in existing_indexes:
    pc.create_index_for_model(
        name=DENSE, cloud="aws", region="us-east-1",
        # "Integrated embedding": Pinecone will embed text FOR us.
        embed={"model": "llama-text-embed-v2",     # Pinecone-hosted embedder (the Romantic)
               "field_map": {"text": "bio"}}       # ← embed the 'bio' field of each record
    )
else:
    print("Index 'dilse-dense' already exists. Skipping creation.")

# Creating an index is async — poll until Pinecone reports it's ready to use
while not pc.describe_index(DENSE).status["ready"]: time.sleep(1)

dense = pc.Index(DENSE)   # a handle we'll use to upsert/search this index
print("Dense index ready ✅")


Index 'dilse-dense' already exists. Skipping creation.
Dense index ready ✅


## Cell 3 — Upsert: send TEXT, get vectors (you never see them)
Records go up as plain dicts. Pinecone embeds `bio` server-side; the other fields become filterable metadata.
*(96 records per call is the integrated-embedding limit — hence the tiny batching loop. Batching lesson #1, for free.)*

In [11]:
NS = "users"   # "namespace" — a logical partition inside the index (like a table/collection)

# Pinecone's integrated-embedding upsert accepts at most 96 records per call,
# so we upload the 400 users in batches of 96.
for i in range(0, len(users), 96):
    dense.upsert_records(namespace=NS,records=users[i:i+96])   # sends plain dicts; Pinecone embeds 'bio' server-side

time.sleep(10)   # give the async embedding a moment (embedding happens in the background)
print(dense.describe_index_stats())   # confirms how many vectors are now stored


DescribeIndexStatsResponse(dimension=1024, total_vector_count=400, metric='cosine', namespaces=1)


# 🅲 SECTION C — BASIC SEMANTIC SEARCH (AND WHERE IT FAILS)

## Cell 4 — Search = 5 lines. The Romantic at her best.
🎲 **CHAT BET:** *"Query: 'someone adventurous who loves trekking'. Will the top hit contain the word 'adventurous'? YES/NO."*
(Either answer teaches: matches come by MEANING — 'travel junkie, 14 countries' can win without the word.)

In [15]:
def search(index, text, k=5, flt=None):
    """Run a semantic/keyword search against `index`.
    text : the natural-language query
    k    : how many results to return (top_k)
    flt  : optional metadata filter dict, e.g. {"college": "IIT Bombay"}
    Returns a list of (profile_id, score, bio) tuples.
    """
    q = {"inputs": {"text": text}, "top_k": k}
    if flt: q["filter"] = flt          # attach the filter only if one was given


    res = index.search(namespace=NS,
                       query=q,
                       fields=["bio"])
    return [(h["id"],
             round(h["score"], 4),
             h["fields"]["bio"]
             ) for h in res["result"]["hits"]]

def show(hits, star=()):
    """Pretty-print search results. `star` is a set of profile ids to mark with ⭐."""
    for r, (pid, s, bio) in enumerate(hits, 1):
        print(f"{r:>2}. {pid}  {s}{'  ⭐' if pid in star else ''}\n    {bio[:105]}")

# Try a plain semantic search: notice the query text doesn't need to match
# the bio word-for-word — matches come from MEANING, not exact keywords.
show(search(dense, "someone adventurous who loves trekking"))


 1. P0187  0.3696
    Architect, ICT Mumbai 2022 batch. Tamil speaking, lives in Andheri. Adventurous and loves trekking in the
 2. P0220  0.3566
    Architect, Mumbai University 2022 batch. Hindi speaking, lives in Thane. Adventurous and loves trekking i
 3. P0093  0.3558
    Data scientist, IIT Bombay 2022 batch. Konkani speaking, lives in Bandra. Adventurous and loves trekking 
 4. P0347  0.3555
    Doctor, IIT Bombay 2017 batch. Hindi speaking, lives in Andheri. Adventurous and loves trekking in the sa
 5. P0195  0.3527
    Software engineer at tcs, VJTI 2020 batch. Gujarati speaking, lives in Dadar. Adventurous and loves trekk


## Cell 4b — 🎫 CRISIS #2, live: the Romantic meets a biodata query
🎲 **CHAT BET:** *"'IIT Bombay 2019 batch, Marathi, lives in Dadar' — of the top 5, how many match ALL FOUR facts? Type 0–5."*
The ✓/✗ table does the arguing. Expected: near-misses (right college, wrong batch...). The Romantic hears **vibes, not facts.**

In [18]:
CRISIS2 = "IIT Bombay 2019 batch, Marathi speaking, lives in Dadar"

def fact_check(hits):
    """Print a ✓/✗ table showing whether each of the 4 required facts

    (college, batch, language, locality) literally appears in each hit's bio."""
    print(f"{'rank':<5}{'id':<7}{'IIT-B':<7}{'2019':<7}{'Marathi':<9}{'Dadar':<7}")

    for r, (pid, s, bio) in enumerate(hits, 1):
        f = ["✓" if w in bio else "✗" for w in ("IIT Bombay", "2019", "Marathi", "Dadar")]
        print(f"{r:<5}{pid:<7}{f[0]:<7}{f[1]:<7}{f[2]:<9}{f[3]:<7}{' ⭐ PERFECT' if f==['✓']*4 else ''}")

# Run the biodata-style query through PURE semantic search and check the facts.
# Expect near-misses: dense search chases "vibe", not literal facts.
#show(search(dense, CRISIS2))
fact_check(search(dense, CRISIS2))
print("\n(The perfect profile P0001 exists. Did the Romantic even surface it?)")


 1. P0388  0.5262
    Doctor, SPIT 2019 batch. Marathi speaking, lives in Dombivli. Classical music and long walks at marine dr
 2. P0001  0.5193
    Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal
 3. P0157  0.5171
    Chartered accountant, SPIT 2019 batch. Marathi speaking, lives in South Bombay. Sorted but chill, big on 
 4. P0177  0.517
    Doctor, IIT Bombay 2019 batch. Gujarati speaking, lives in Thane. Fun-loving but settled, enjoys quiet we
 5. P0218  0.516
    Graphic designer, IIT Bombay 2019 batch. Gujarati speaking, lives in Bandra. Foodie who hunts for the bes
rank id     IIT-B  2019   Marathi  Dadar  
1    P0388  ✗      ✓      ✓        ✗      
2    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT
3    P0157  ✗      ✓      ✓        ✗      
4    P0177  ✓      ✓      ✗        ✗      
5    P0218  ✓      ✓      ✗        ✗      

(The perfect profile P0001 exists. Did the Romantic even surface it?)


# 🅳 SECTION D — PRODUCTION FIX #1: METADATA FILTERS

## Cell 5 — Production fix #1: facts don't belong in embeddings. They belong in FILTERS.
**ONE LINE.** This is the single most production-real lesson of the day: structured facts → metadata filter; the embedding only handles the fuzzy part.

🎲 **CHAT BET:** *"Same search + a metadata filter. Perfect matches in the top 5: how many now?"*

In [17]:
# Split the query into two parts:
#   1. the FUZZY part ("looking for a genuine connection") → handled by the embedding
#   2. the EXACT facts (college/batch/lang/locality)        → handled by a metadata filter (a WHERE clause)
hits = search(dense, "looking for a genuine connection",         # the fuzzy part
              flt={"college": "IIT Bombay",
                   "batch": 2019,        # ← the facts, as a WHERE clause
                   "lang": "Marathi",
                   "locality": "Dadar"})
fact_check(hits)   # now every returned hit is GUARANTEED to satisfy all 4 facts
print("\nRemember Session 3's trick question — 'similar AND filtered'? BOTH, in one call. That was the answer.")


rank id     IIT-B  2019   Marathi  Dadar  
1    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT

Remember Session 3's trick question — 'similar AND filtered'? BOTH, in one call. That was the answer.


In [20]:
# P0001  0.5193 Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal

hits = search(dense, "someone adventurous who loves trekking",         # the fuzzy part
              flt={"college": "IIT Bombay",
                   "batch": 2019,        # ← the facts, as a WHERE clause
                   "lang": "Marathi",
                   "locality": "Dadar"})
fact_check(hits)
print("\n(The perfect profile P0001 exists. Did the Romantic even surface it?)")

rank id     IIT-B  2019   Marathi  Dadar  
1    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT

(The perfect profile P0001 exists. Did the Romantic even surface it?)


# 🅴 SECTION E — KEYWORD SEARCH (SPARSE / BM25 INDEX)

## Cell 5b — But what about keyword *relevance*? Meet the Aunty (a real sparse index)
Filters are binary: in or out. Sometimes you want exact words to *boost ranking*, not gatekeep — "golden retriever" should RANK higher, not exclude everyone else. That's a **sparse index**: Pinecone hosts a BM25-family model (`pinecone-sparse-english-v0`). Same API, second index. Zero math.

In [23]:
SPARSE = "dilse-sparse"   # name of our keyword ("sparse vector") index

if not pc.has_index(SPARSE):
    pc.create_index_for_model(
        name=SPARSE, cloud="aws", region="us-east-1",
        embed={"model": "pinecone-sparse-english-v0",   # hosted BM25-style lexical model (the Aunty)
               "field_map": {"text": "bio"}}
    )
while not pc.describe_index(SPARSE).status["ready"]: time.sleep(1)

sparse = pc.Index(SPARSE)
# Upload the same 400 users into the sparse index, in batches of 96 again.
for i in range(0, len(users), 96):
    sparse.upsert_records(namespace=NS,records= users[i:i+96])
time.sleep(10); print("Sparse index ready ✅")


Sparse index ready ✅


## Cell 5c — The Aunty's superpower... and her blind spot, back-to-back
🎲 **CHAT BET 1:** *"Aunty on the biodata query — perfect match at #1: YES/NO?"*
🎲 **CHAT BET 2:** *"Query 'fun-loving but settled'. P0002 says 'sorted but chill' — ZERO shared words. Will Aunty find P0002 in top 5? Will the Romantic?"*

In [26]:
# 1) On the biodata query, keyword search (Aunty) should nail exact facts easily.
print("=== AUNTY (sparse) on the biodata query ===")
# CRISIS2 = "IIT Bombay 2019 batch, Marathi speaking, lives in Dadar"CRISIS2 = "IIT Bombay 2019 batch, Marathi speaking, lives in Dadar"#
fact_check(search(sparse, CRISIS2))

# 2) On a paraphrase query, keyword search struggles because P0002's bio shares
#    ZERO exact words with the query ("sorted but chill" vs "fun-loving but settled").
# P0002 → Architect, Mumbai University 2014 batch. Tamil speaking, lives in Bandra. Sorted but chill, big on family, small on drama. Occasional smoker.
print("\n=== AUNTY on: 'fun-loving but settled' ===")
show(search(sparse, "fun-loving but settled, enjoys a quiet weekend"), star={"P0002"})

# Dense (Romantic) should still find it, because it matches on MEANING, not words.
print("\n=== ROMANTIC (dense) on the same query ===")
show(search(dense, "fun-loving but settled, enjoys a quiet weekend"), star={"P0002"})

print("\nComplementary blind spots → hire both. That's hybrid.")


=== AUNTY (sparse) on the biodata query ===
rank id     IIT-B  2019   Marathi  Dadar  
1    P0001  ✓      ✓      ✓        ✓       ⭐ PERFECT
2    P0050  ✓      ✗      ✓        ✓      
3    P0166  ✓      ✓      ✓        ✗      
4    P0107  ✓      ✗      ✓        ✗      
5    P0374  ✓      ✗      ✓        ✗      

=== AUNTY on: 'fun-loving but settled' ===
 1. P0299  12.2463
    Graphic designer, NMIMS 2014 batch. Hindi speaking, lives in Byculla. Fun-loving but settled, enjoys quie
 2. P0255  12.0908
    Architect, Mumbai University 2020 batch. Hindi speaking, lives in Powai. Fun-loving but settled, enjoys q
 3. P0164  12.0863
    Teacher, Mumbai University 2016 batch. Konkani speaking, lives in Andheri. Fun-loving but settled, enjoys
 4. P0226  12.0694
    Architect, Mumbai University 2015 batch. Hindi speaking, lives in Bandra. Fun-loving but settled, enjoys 
 5. P0173  12.0547
    Architect, VJTI 2020 batch. Marathi speaking, lives in Dadar. Fun-loving but settled, enjoys quiet weeken

# 🅵 SECTION F — HYBRID SEARCH (DENSE + SPARSE, MERGED WITH RRF)

## Cell 6 — Hybrid: query both, merge with RRF (the pattern production actually uses)
Two indexes, one merge function. **Reciprocal Rank Fusion** — trust *ranks*, not raw scores (which live on different scales). α still exists: it weights how much you trust each matchmaker.

🎲 **CHAT GAME (α auction):** *"Type ONE number 0–1: your α for a matrimony app. Now for casual dating."* Run with the room's median, then re-run at 0.9 and 0.1 — predict what breaks each time.

In [27]:
def hybrid(text, alpha=0.5, k=5):
    """Combine dense + sparse search results using Reciprocal Rank Fusion (RRF).
    alpha : how much to trust the dense (semantic) results vs sparse (keyword) results.
            alpha=1.0 -> pure dense,
            alpha=0.0 -> pure sparse,
            0.5 -> balanced.
    RRF works on RANKS (1st place, 2nd place, ...) rather than raw scores,
    because dense and sparse scores live on different, incomparable scales.
    """
    d = [pid for pid, _, _ in search(dense,  text, k=25)]   # top 25 ids from dense search
    s = [pid for pid, _, _ in search(sparse, text, k=25)]   # top 25 ids from sparse search
    score = {}
    # RRF formula: each result contributes 1/(60 + rank) to its fused score,
    # weighted by alpha for dense and (1 - alpha) for sparse.
    for rank, pid in enumerate(d): score[pid] = score.get(pid, 0) + alpha       / (60 + rank)
    for rank, pid in enumerate(s): score[pid] = score.get(pid, 0) + (1 - alpha) / (60 + rank)
    top = sorted(score, key=score.get, reverse=True)[:k]   # highest fused score first
    bio = {u["_id"]: u["bio"] for u in users}
    return [(pid, round(score[pid], 5), bio[pid]) for pid in top]

MIXED = "IIT Bombay 2019 Marathi Dadar, adventurous trekking type"   # facts AND vibes in one query
# Sweep alpha from pure-sparse (0.0) to pure-dense (1.0) and see the ranking shift.
for a in (0.0, 0.5, 1.0):
    print(f"\n──── α = {a}  ({'pure AUNTY' if a==0 else 'pure ROMANTIC' if a==1 else 'diplomat'}) ────")
    show(hybrid(MIXED, alpha=a, k=3), star={"P0001"})



──── α = 0.0  (pure AUNTY) ────
 1. P0001  0.01667  ⭐
    Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal
 2. P0050  0.01639
    Doctor, IIT Bombay 2015 batch. Marathi speaking, lives in Dadar. Travel junkie, 14 countries and counting
 3. P0166  0.01613
    Architect, IIT Bombay 2019 batch. Marathi speaking, lives in Bandra. Stand-up comedy fan, drags friends t

──── α = 0.5  (diplomat) ────
 1. P0001  0.01591  ⭐
    Data scientist, IIT Bombay 2019 batch. Marathi speaking, lives in Dadar. Loves dogs, volunteers at animal
 2. P0093  0.01505
    Data scientist, IIT Bombay 2022 batch. Konkani speaking, lives in Bandra. Adventurous and loves trekking 
 3. P0387  0.01317
    Architect, ICT Mumbai 2019 batch. Marathi speaking, lives in Dadar. Classical music and long walks at mar

──── α = 1.0  (pure ROMANTIC) ────
 1. P0005  0.01667
    Graphic designer, NMIMS 2019 batch. Konkani speaking, lives in Thane. Adventurous and loves trekk

# 🅶 SECTION G — SCALING: VECTOR COMPRESSION (QUANTIZATION)

---
## ☕ BREAK 1 · leave on screen: *"After break: the investor's 300 GB email — and why the fix is ZERO lines of your code."*
---

## Cell 7 — 🎫 CRISIS #1: The Burn-Rate math (do it WITH the class in chat)
Then the punchline: **quantization**. On screen next: the by-hand chat race — the only 'code' is the one your BRAIN runs.

In [28]:
# How much RAM would 100M full-precision vectors take, and how much after compression?
N, dim, fbytes = 100_000_000, 1024, 4        # llama-text-embed-v2 default: 1024 dims
# fbytes = 4 because each dimension is stored as a 32-bit (4-byte) float
print(f"{N:,} users × {dim} dims × {fbytes} B = {N*dim*fbytes/1e9:,.0f} GB of RAM. Investor: 😡")
# Product quantization (PQ) can shrink storage by roughly 32× with small accuracy loss
print(f"After ~32× product quantization: {N*dim*fbytes/1e9/32:,.0f} GB. Investor: 😍")


100,000,000 users × 1024 dims × 4 B = 410 GB of RAM. Investor: 😡
After ~32× product quantization: 13 GB. Investor: 😍


## Cell 7b — ⭐ PQ by hand (the chat race — this IS the lesson)
Paste in chat: *"Compress this person! For each chunk pick the closest TYPE (0–3). Type your 4 codes, e.g. `0 3 1 2`. GO!"*

```
PROFILE VECTOR (8 numbers, chunked in 2s):
    [ 0.9, 0.1 | 0.8, 0.2 | 0.1, 0.9 | 0.5, 0.5 ]

CODEBOOK OF TYPES (valid for every chunk):
    Type 0: (0.9, 0.1)    Type 1: (0.1, 0.9)
    Type 2: (0.5, 0.5)    Type 3: (0.8, 0.3)
```

**Reveal:** `0 3 1 2` · 32 bytes → 1 byte = **32×** · decode chunk 2: you stored (0.8, 0.2) as (0.8, 0.3) — that gap is **quantization error**. Aunty remembers you as your *type*, not as YOU. Same devil as yesterday's ANN deal — new contract.

**Then the production truth (say it slowly):** *"And here is the best part: in Pinecone, YOU WRITE NONE OF THIS. Compression, codebooks, retraining when the user base drifts — managed, under the hood. The knobs you learned (m = chunks, nbits = types-per-chunk) are what you'd tune in FAISS if you ever build the engine yourself — and what you now READ fluently when Pinecone's docs mention them."*

In [33]:
# Optional under-the-hood peek (FAISS) — for the curious, NOT for production. Flip to run.
SHOW_ENGINE_ROOM = True   # set to True to actually run the FAISS demo below
if SHOW_ENGINE_ROOM:
    !pip -q install faiss-cpu numpy
    import faiss, numpy as np
    X = np.random.rand(100_000, 128).astype("float32")   # 100k fake 128-dim vectors
    pq = faiss.IndexPQ(128, 16, 8)          # 16 chunks, 2^8 = 256 types per chunk
    pq.train(X); pq.add(X)                  # learn the "codebook", then compress+store X
    print(f"raw: {X.nbytes/1e6:.0f} MB  →  PQ codes: {100_000*16/1e6:.1f} MB  (32×)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.5 MB/s eta 0:00:00
raw: 51 MB  →  PQ codes: 1.6 MB  (32×)


# 🅷 SECTION H — EVALUATION METRICS (HOW GOOD IS THE SEARCH?)

## Cell 8 — 🎫 CRISIS #3a: prove matching is good. With numbers.
These five tiny functions ARE what practitioners write (or import from an evals library). First the chat race on the card, then the same metrics on OUR app — since we generated the users, we KNOW who's truly compatible (real teams buy this ground truth with human labels).

🎲 **CHAT RACE:** *"Shown [A,B,C,D,E], compatible {B,E,G}. Compute P@5, R@5, MRR@5 — fastest correct wins."*

In [39]:
import math

# --- Standard information-retrieval metrics -----------------------------
# ranked : list of ids in the order the search engine returned them ["A","B","C","D","E"]
# rel    : the SET of ids that are actually "relevant"/correct (ground truth) {"B","E","G"}
# k      : how many top results to consider 5

def precision_at_k(ranked, rel, k):
    # Of the top k results, what fraction are actually relevant? =2/5 =0.4
    return sum(x in rel for x in ranked[:k]) / k

def recall_at_k(ranked, rel, k):
    # Of ALL relevant items, what fraction did we manage to surface in the top k? =2/3 =0.67
    return sum(x in rel for x in ranked[:k]) / max(1, len(rel))

def mrr_at_k(ranked, rel, k):
    # Mean Reciprocal Rank: 1 / (rank of the FIRST relevant hit). Rewards finding 1/2 =0.5
    # a good match EARLY — 0 if there's no relevant hit in the top k.
    return next((1/i for i, x in enumerate(ranked[:k], 1) if x in rel), 0.0)

def ap_at_k(ranked, rel, k):
    # Average Precision: average of precision@i, evaluated only at each rank
    # where a relevant item appears. Divides by the TOTAL number of relevant
    # items (not just the ones found), so missed matches still hurt the score.
    hits = ap = 0
    for i, x in enumerate(ranked[:k], 1):
        if x in rel: hits += 1; ap += hits / i
    return ap / max(1, len(rel))

def ndcg_at_k(ranked, gains, k):
    # Normalized Discounted Cumulative Gain: like precision, but supports
    # "graded" relevance (some matches are better than others) and discounts
    # items that appear lower in the ranking.
    dcg  = sum(gains.get(x, 0) / math.log2(i+1) for i, x in enumerate(ranked[:k], 1))
    idcg = sum(g / math.log2(i+1) for i, g in enumerate(sorted(gains.values(), reverse=True)[:k], 1))
    return dcg / idcg if idcg else 0.0

# --- Quick sanity check on a toy example, doable by hand -----------------
card_ranked, card_rel = ["A","B","C","D","E"], {"B","E","G"}
print(f"CARD → "
      f"P@5={precision_at_k(card_ranked, card_rel, 5):.2f}"
      f"R@5={recall_at_k(card_ranked, card_rel, 5):.2f}  "
      f"MRR@5={mrr_at_k(card_ranked, card_rel, 5):.2f}  "
      f"AP@5={ap_at_k(card_ranked, card_rel, 5):.2f}"
      )
print("Why does AP divide by 3, not 2? The match we FAILED to show (G) must hurt the score.")


CARD → P@5=0.40R@5=0.67  MRR@5=0.50  AP@5=0.30
Why does AP divide by 3, not 2? The match we FAILED to show (G) must hurt the score.


## Cell 8b — The metrics on OUR app
🎲 **CHAT BET:** *"Query persona: non-smoker · loves dogs · Dadar. Guess the rank of the FIRST truly compatible profile in plain dense top-20."*
**Talking point on output:** decent recall, painful MRR = Crisis #3, measured. "How far does she scroll before the first real match? MRR is the patience-meter."

In [ ]:
QUERY3 = "non-smoker who loves dogs, based in Dadar"

# Ground truth: since WE generated the data, we know exactly which profiles
# are truly compatible with this query (a real app would need human labels for this).
truly = {u["_id"] for u in users
         if u["smoke"] == "non-smoker" and "dog" in u["vibe"] and u["locality"] == "Dadar"}
print(f"Ground truth: {len(truly)} truly compatible → {sorted(truly)}\n")

# Run plain dense search and see how it ranks against the ground truth.
stage1 = [pid for pid, _, _ in search(dense, QUERY3, k=20)]
first = next((i for i, x in enumerate(stage1, 1) if x in truly), ">20")   # rank of first true match
print("Dense top-20:", stage1[:10], "…")
print(f"P@5={precision_at_k(stage1, truly, 5):.2f}  R@10={recall_at_k(stage1, truly, 10):.2f}  "
      f"MRR@10={mrr_at_k(stage1, truly, 10):.2f}  |  first TRUE match at rank {first}")


# 🅸 SECTION I — RERANKING (TWO-STAGE RETRIEVAL)

---
## ☕ BREAK 2 · teaser: *"The rank-12 fix is four lines. Not four hundred. Four."*
---

## Cell 9 — 🎫 CRISIS #3b: Reranking = ONE parameter on the SAME call
**Say:** *"Stage 1 swipes (fast, 400 → 25 maybes). Stage 2 is the coffee date: a cross-encoder reads the query and each profile TOGETHER — like your brain did when you insta-picked the golden-retriever profile. In production, that judge is hosted. You add a `rerank` block. That's the entire diff."*

🎲 **CHAT BET:** *"After reranking: MRR@10 UP/DOWN/SAME? P@5?"*

In [ ]:
# Stage 1 (retrieval): fetch 25 candidates fast, using the same dense search as before.
# Stage 2 (rerank): a "rerank" block asks a hosted cross-encoder model to
# re-read the query against EACH candidate profile TOGETHER, then re-sort
# the top 25 down to the best 10. This single extra parameter is the whole diff.
res = dense.search(
    namespace=NS,
    query={"inputs": {"text": QUERY3}, "top_k": 25},      # Stage 1: the swipe
    fields=["bio"],
    rerank={"model": "bge-reranker-v2-m3",                # Stage 2: the coffee date (hosted)
            "top_n": 10,
            "rank_fields": ["bio"]}
)
stage2 = [h["_id"] for h in res["result"]["hits"]]   # ids, now reranked

# Compare the same metrics BEFORE (stage1, plain retrieval) vs AFTER (stage2, reranked).
print(f"{'metric':<8}{'before':>9}{'after':>9}")
for name, fn, k in [("MRR@10", mrr_at_k, 10), ("P@5", precision_at_k, 5), ("AP@10", ap_at_k, 10)]:
    print(f"{name:<8}{fn(stage1, truly, k):>9.2f}{fn(stage2, truly, k):>9.2f}")
print("\nRank 12 → top 3. Prasad's cousin gets her match. 🎫#3 CLOSED ✅")
print("Anti-confusion: HYBRID blends scores AT retrieval. RERANK is a second pass AFTER. Party vs shortlist.")


# 🅹 SECTION J — PRODUCTION OPS: LATENCY & CACHING

## Cell 10 — 🎫 CRISIS #4: Launch night. Latency percentiles in 10 lines.
First show the incident table (below) and let the room diagnose in chat. THEN run the timer on our real index.

| QPS | p50 | p95 | **p99** |
|---|---|---|---|
| 150,000 | 5 ms | 50 ms | **1,200 ms** |

🎲 **CHAT:** *"Prasad says '150K QPS is AMAZING!' What do you tell him? Launch-ready: YES/NO + why."*
**Land on:** p99 = the 1% staring at a spinner for 1.2 s mid-swipe ≈ 1,500 ruined moments/second at that QPS. **Averages flirt; tails tell the truth.** *(Note: our numbers below include network to the cloud — that's honest end-to-end latency, the number users feel.)*

In [ ]:
import numpy as np
lat = []   # will hold 60 latency measurements, in milliseconds

# Run the same search 60 times and time each call end-to-end (network + embed + search).
for _ in range(60):
    t0 = time.perf_counter()
    search(dense, "dog lover in Dadar", k=10)
    lat.append((time.perf_counter() - t0) * 1000)   # convert seconds → milliseconds

# Percentiles matter more than the average: p99 tells you how slow it is
# for the UNLUCKIEST 1% of requests — that's what "the tail" means.
p50, p95, p99 = np.percentile(lat, [50, 95, 99])
print(f"end-to-end latency  p50={p50:.0f} ms   p95={p95:.0f} ms   p99={p99:.0f} ms")
print("\nIn production you don't hand-roll this: the Pinecone console charts QPS + percentiles live,")
print("and teams pipe them to Prometheus/Datadog. [→ 60-second console tour now]")


## Cell 10b — The free lunch: caching (8 lines, and yes, real apps do exactly this)
🎲 **CHAT BET:** *"'matches near Dadar' is asked 5,000×/day. Cache hit vs fresh search — how many × faster? 5 / 50 / 500?"*
Gotchas to name after: **invalidation** (new users join → stale cache) and **near-duplicates** ('matches near Dadar' vs 'Dadar-area matches'). In prod: same dict, but it's called Redis.

In [ ]:
cache = {}   # in-memory cache: {normalized query text -> search results}
             # in a real production system this dict would be Redis instead

def cached_search(text, k=5):
    key = text.strip().lower()   # normalize so "Dadar" and "dadar " hit the same cache entry
    if key not in cache:
        cache[key] = search(dense, text, k)   # cache MISS: actually call the search API
    return cache[key]                          # cache HIT: instant, no network call

# First call: cache miss (has to embed + search over the network).
t0 = time.perf_counter(); cached_search("matches near Dadar"); miss = (time.perf_counter()-t0)*1000
# Second call, same text: cache hit (just a dict lookup).
t0 = time.perf_counter(); cached_search("matches near Dadar"); hit  = (time.perf_counter()-t0)*1000

print(f"MISS {miss:8.2f} ms  (embed + search, over the network)")
print(f"HIT  {hit:8.4f} ms  →  {miss/max(hit, 1e-9):,.0f}× faster. 🎫#4 CLOSED ✅")


# 🅺 SECTION K — WRAP-UP & CLEANUP

## The Close (no code)
Point at the four crises in chat; the class closes each in ONE word:
**#1** quantization (0 lines — managed) · **#2** metadata filter + sparse (1 line) · **#3** rerank (4 lines) · **#4** percentiles + caching (~10 lines).

> *"Yesterday the algorithm found ONE person a match at a pink café in Dadar. Today it does it for 100 million — affordably, exactly, provably, fast even for the unluckiest 1%... in about 25 lines of application code. That's the difference between a love story and a company. And THAT is why managed vector databases exist."*

**Homework:** invent your own query persona + ground-truth rule; report P@5 and MRR@10 before vs after `rerank=`. One screenshot, two numbers.
**Cleanup (avoid charges):** `pc.delete_index("dilse-dense"); pc.delete_index("dilse-sparse")`